# 🧠 Chain of Thought (CoT) با LiteLLM و Ollama

در این نوت‌بوک مفهوم **Chain of Thought Prompting** را به صورت گام به گام یاد می‌گیریم.

## فهرست مطالب
 -  مقدمه: CoT چیست؟
 -  آزمایش پایه: بدون CoT vs با CoT
 - Zero-Shot CoT
 - Few-Shot CoT
 - Self-Consistency CoT
 - مدل‌های Reasoning (deepseek-r1)
 - مقایسه مدل‌ها

---
**مدل‌های موجود:**
- `gemma-4-26B:latest` | `gemma4:e4b` | `gemma3:4b`
- `deepseek-r1:8b` | `deepseek-coder:6.7b`
- `llama3.1:8b` | `qwen3:14b`
- `gpt-oss:20b` | `translategemma:4b`

## 📌 گام ۱: CoT چیست؟

**Chain of Thought (CoT)** یک تکنیک prompting است که به مدل زبانی می‌گوید **قبل از پاسخ نهایی، مراحل استدلال خود را قدم به قدم بنویسد**.

```
بدون CoT:  سوال → پاسخ
با CoT:    سوال → گام ۱ → گام ۲ → گام ۳ → پاسخ
```

### چرا CoT مهم است؟
- بهبود دقت در مسائل پیچیده (ریاضی، منطق، استدلال)
- شفافیت در تصمیم‌گیری مدل
- کاهش خطاهای محاسباتی
- پایه‌ی روش‌های پیشرفته‌تر مثل ReAct و ToT

## 📌 گام ۲: نصب و راه‌اندازی

In [1]:
import litellm
from litellm import completion
import json
import time


MODELS = {
    "gemma_26b": "ollama/gemma-4-26B:latest",
    "gemma_10b": "ollama/gemma4:e4b",
    "deepseek_r1": "ollama/deepseek-r1:8b",   # مدل reasoning
    "deepseek_coder": "ollama/deepseek-coder:6.7b",
    "llama31": "ollama/llama3.1:8b",
    "gemma_4b": "ollama/gemma3:4b",
    "qwen3": "ollama/qwen3:14b",
    "gpt_oss": "ollama/gpt-oss:20b",
}

DEFAULT_MODEL = MODELS["llama31"]

print("✅ LiteLLM آماده است")
print(f"🤖 مدل پیش‌فرض: {DEFAULT_MODEL}")

✅ LiteLLM آماده است
🤖 مدل پیش‌فرض: ollama/llama3.1:8b


In [5]:
def ask(prompt: str, model: str = DEFAULT_MODEL, system: str = None, temperature: float = 0.1) -> str:
    """
    یک تابع ساده برای ارسال prompt به مدل
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    start = time.time()
    response = completion(
        model=model,
        messages=messages,
        temperature=temperature)
    elapsed = time.time() - start
    
    content = response.choices[0].message.content
    print(f"⏱️ زمان پاسخ: {elapsed:.1f}s | مدل: {model.split('/')[-1]}")
    return content


def print_response(label: str, response: str):
    """نمایش زیبای پاسخ"""
    print(f"\n{'='*60}")
    print(f"📋 {label}")
    print(f"{'='*60}")
    print(response)
    print(f"{'='*60}\n")


# تست اتصال
test = ask("سلام! یه جمله کوتاه بگو.")
print_response("تست اتصال", test)

⏱️ زمان پاسخ: 0.9s | مدل: llama3.1:8b

📋 تست اتصال
خوشحالیم که با شما صحبت می کنیم.



## 📌 گام ۳: بدون CoT در مقابل با CoT

بیایید با یک مسئله ریاضی ساده شروع کنیم و تفاوت را ببینیم.

In [21]:
# مسئله آزمایشی
PROBLEM = """
علی ۵ تا سیب داشت. ۳ تا از سیب ها را مریم ازش دزدید، بعد ۷ تا خرید.
سپس نصف سیب‌هایش را به دوستش داد.
الان علی چند تا سیب دارد؟
"""

# ---- روش ۱: بدون CoT ----
prompt_without_cot = f"""
سوال: {PROBLEM} 


لطفا فقط فقط پاسخ نهایی را بده

جواب:
"""

response_no_cot = ask(prompt_without_cot)
print_response("❌ بدون CoT", response_no_cot)

⏱️ زمان پاسخ: 0.5s | مدل: llama3.1:8b

📋 ❌ بدون CoT
۵ + ۷ - ۳ = ۹



In [27]:
# ---- روش ۲: با CoT ----
prompt_with_cot = f"""
سوال: {PROBLEM}

مرحله به مرحله فکر کن و هر قدم را توضیح بده، سپس جواب نهایی را بنویس.

مراحل حل:
"""

response_with_cot = ask(prompt_with_cot)
print_response("✅ با CoT", response_with_cot)

⏱️ زمان پاسخ: 4.2s | مدل: llama3.1:8b

📋 ✅ با CoT
### مراحل حل:

**مرحله 1:** ابتدا تعداد سیب‌های علی را که در ابتدا داشتیم می‌گوییم. او 5 تا سیب داشت.

**مرحله 2:** مریم 3 تا از سیب‌ها را دزدید، پس حالا علی 5 - 3 = 2 سیب دارد.

**مرحله 3:** سپس علی 7 تا سیب خرید، بنابراین حالا تعداد سیب‌هایش به 2 + 7 = 9 می‌رسد.

**مرحله 4:** نهایتاً، علی نصف سیب‌های خود را به دوستش داد. اگر Ali 9 سیب دارد و نیمی از آن‌ها را به دوستش می‌دهد، پس او 9/2 = 4.5 سیب خواهد داشت.



## 📌 گام ۴: Zero-Shot CoT

ساده‌ترین روش CoT: فقط اضافه کردن عبارت **"Let's think step by step"** یا معادل فارسی آن.

> 📄 مقاله اصلی: *Large Language Models are Zero-Shot Reasoners* (Kojima et al., 2022)

In [29]:
def zero_shot_cot(question: str, model: str = DEFAULT_MODEL, lang: str = "fa") -> str:
    """
    Zero-Shot CoT: فقط با یک عبارت جادویی!
    """
    magic_phrases = {
        "en": "Let's think step by step.",
        "fa": "بیا گام به گام فکر کنیم.",
    }
    
    magic = magic_phrases.get(lang, magic_phrases["en"])
    
    prompt = f"{question}\n\n{magic}"
    return ask(prompt, model=model)


# آزمایش ۱: مسئله ریاضی
math_q = "اگر یک قطار با سرعت ۱۲۰ کیلومتر در ساعت حرکت کند و ساعت 5 راه افتاده و ساعت 7:30 رسیده، چه مسافتی را طی کرده؟"

result = zero_shot_cot(math_q)
print_response("Zero-Shot CoT - مسئله ریاضی", result)

⏱️ زمان پاسخ: 4.4s | مدل: llama3.1:8b

📋 Zero-Shot CoT - مسئله ریاضی
ابتدا باید فاصله زمانی بین ساعت 5 و 7:30 را پیدا کنیم. این فاصله 2 ساعت و 30 دقیقه است. اگر می‌خواهیم این زمان را به ساعت تبدیل کنیم، باید آن را به 2 ساعت و 30/60 ساعت (یا 2 + 0.5) تغییر دهیم.

حال که داریم با یک قطار با سرعت 120 کیلومتر در ساعت کار می‌کنیم، برای پیدا کردن مسافتی که طی کرده است، باید این سرعت را با زمان طی شده multiplie کنیم:

مسافت = سرعت × زمان
مسافت = 120 × (2 + 0.5)
مسافت = 120 × 2.5
مسافت = 300 کیلومتر

بنابراین، قطار در 2 ساعت و 30 دقیقه مسیر خود را تا 300 کیلومتر طی کرده است.

پاسخ نهایی: 300



In [33]:
# آزمایش ۲: مسئله منطقی
logic_q = """
سه برادر داریم: احمد، باقر، حسن.
احمد از باقر بزرگ‌تر است.
حسن از احمد جوان‌تر است اما از باقر بزرگ‌تر.
کدام برادر بزرگترین است؟
"""

result_logic = zero_shot_cot(logic_q)
print_response("Zero-Shot CoT - مسئله منطقی", result_logic)

⏱️ زمان پاسخ: 0.7s | مدل: llama3.1:8b

📋 Zero-Shot CoT - مسئله منطقی
بزرگ‌ترین برادر Ahmad است.



In [17]:
# مقایسه Zero-Shot CoT انگلیسی vs فارسی
q = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"

print("🔵 با عبارت انگلیسی:")
r_en = zero_shot_cot(q, lang="en")
print(r_en[:500])

print("\n🟢 با عبارت فارسی:")
r_fa = zero_shot_cot(q, lang="fa")
print(r_fa[:500])

🔵 با عبارت انگلیسی:
⏱️ زمان پاسخ: 5.1s | مدل: llama3.1:8b
To find out how many tennis balls Roger has now, we need to follow these steps:

1. First, let's calculate the total number of tennis balls in each can: 
   Each can has 3 tennis balls.

2. Now, let's determine the total number of cans Roger bought:
   He buys 2 more cans of tennis balls.

3. Next, let's find out how many tennis balls he got from these new cans:
   Since each can has 3 tennis balls and he bought 2 cans, we multiply the number of cans by the number of tennis balls in each can: 
 

🟢 با عبارت فارسی:
⏱️ زمان پاسخ: 6.1s | مدل: llama3.1:8b
گام 1: ابتدا تعداد Tennis Balls را که Roger دارد، مشخص می‌کنیم. او 5 Tennis Ball دارد.
گام 2: سپس تعداد Tennis Balls را که در هر کانت قرار دارد، مشخص می‌کنیم. هر کانت 3 Tennis Ball دارد.
گام 3: حالا باید تعداد Tennis Balls را که Roger خریداری کرده است، مشخص کنیم. او 2 کانت Tennis Ball خریداری کرده است. بنابراین، تعداد Tennis Balls که خریداری کرده است برابر با 2 * 3 = 6 می‌باشد.
گام

## 📌 گام ۵: Few-Shot CoT

در این روش، چند **مثال حل‌شده** به مدل می‌دهیم تا الگو را یاد بگیرد.

> 📄 مقاله اصلی: *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (Wei et al., 2022)

In [24]:
# Few-Shot Examples (نمونه‌های آموزشی)
FEW_SHOT_EXAMPLES = [
    {
        "question": "دانا ۱۰ کتاب داشت. ۴ تا قرض داد و ۶ تا خرید. چند کتاب دارد؟",
        "reasoning": """
گام ۱: دانا ابتدا ۱۰ کتاب دارد.
گام ۲: ۴ کتاب قرض می‌دهد → ۱۰ - ۴ = ۶ کتاب
گام ۳: ۶ کتاب می‌خرد → ۶ + ۶ = ۱۲ کتاب
جواب: ۱۲ کتاب""",
    },
    {
        "question": "یک پارکینگ ۳ ردیف دارد. هر ردیف ۸ جا دارد. ۱۳ ماشین پارک است. چند جا خالی است؟",
        "reasoning": """
گام ۱: کل ظرفیت = ۳ × ۸ = ۲۴ جا
گام ۲: جاهای خالی = ۲۴ - ۱۳ = ۱۱ جا
جواب: ۱۱ جا خالی است""",
    },
]


def few_shot_cot(question: str, examples: list, model: str = DEFAULT_MODEL) -> str:
    """
    Few-Shot CoT: با نمونه‌های آموزشی
    """
    # ساخت prompt با نمونه‌ها
    prompt_parts = []
    
    for i, ex in enumerate(examples, 1):
        prompt_parts.append(f"مثال {i}:")
        prompt_parts.append(f"سوال: {ex['question']}")
        prompt_parts.append(f"استدلال:{ex['reasoning']}")
        prompt_parts.append("")
    
    prompt_parts.append(f"حالا سوال جدید:")
    prompt_parts.append(f"سوال: {question}")
    prompt_parts.append("استدلال:")
    
    full_prompt = "\n".join(prompt_parts)
    return ask(full_prompt, model=model)


# آزمایش Few-Shot CoT
new_q = "یک مغازه روزانه ۱۵۰ محصول می‌فروشد. در ۵ روز چند محصول می‌فروشد؟ اگر هر محصول ۲۰۰۰ تومان باشد، درآمد کل چقدر است؟"

result_few = few_shot_cot(new_q, FEW_SHOT_EXAMPLES)
print_response("Few-Shot CoT", result_few)

⏱️ زمان پاسخ: 3.8s | مدل: llama3.1:8b

📋 Few-Shot CoT
گام ۱: کل فروش = ۱۵۰ × ۵ = ۷۵۰
گام ۲: درآمد کل = ۷۵۰ × ۲۰۰۰ = ۱,۵۰۰۰۰۰۰ تومان
جواب: درآمد کل ۱۵ میلیون تومان است.



## 📌 گام ۶: Self-Consistency CoT

به جای یک پاسخ، **چندین بار** از مدل می‌پرسیم (با دمای مختلف) و **رأی‌گیری** می‌کنیم.

```
سوال → [پاسخ ۱] → استخراج جواب
      → [پاسخ ۲] → استخراج جواب  → رأی‌گیری → جواب نهایی
      → [پاسخ ۳] → استخراج جواب
```

> 📄 مقاله: *Self-Consistency Improves Chain of Thought Reasoning* (Wang et al., 2022)

In [35]:
from collections import Counter
import re

def extract_final_answer(response: str) -> str:
    """
    استخراج جواب نهایی از پاسخ مدل
    ابتدا دنبال فرمت 'جواب نهایی: [عدد]' می‌گردد
    در غیر این صورت آخرین عدد را برمی‌گرداند
    """
    # جستجو برای الگوی 'جواب نهایی: عدد'
    match = re.search(r'جواب نهایی:\s*(\d+)', response)
    if match:
        return match.group(1)
    
    # fallback: آخرین عدد موجود در پاسخ
    numbers = re.findall(r'\d+', response)
    if numbers:
        return numbers[-1]
    
    return "نامشخص"


def self_consistency_cot(
    question: str, 
    n_samples: int = 3, 
    temperature: float = 0.7,
    model: str = DEFAULT_MODEL
) -> dict:
    """
    Self-Consistency CoT
    چندین بار اجرا می‌کند و جواب اکثریت را برمی‌گرداند
    """
    prompt = f"""{question}

بیا گام به گام حل کنیم:

(لطفاً در انتهای پاسخ، جواب نهایی را دقیقاً با این فرمت بنویس:
جواب نهایی: [عدد])"""
    
    responses = []
    answers = []
    
    print(f"🔄 در حال اجرای {n_samples} نمونه...")
    
    for i in range(n_samples):
        print(f"  نمونه {i+1}/{n_samples}...", end="", flush=True)
        response = ask(prompt, model=model, temperature=temperature)
        responses.append(response)
        
        answer = extract_final_answer(response)
        answers.append(answer)
        print(f" → {answer}")
    
    # رأی‌گیری (voting)
    answer_counts = Counter(answers)
    most_common = answer_counts.most_common(1)[0]
    
    return {
        "responses": responses,
        "extracted_answers": answers,
        "vote_result": most_common[0],
        "vote_count": most_common[1],
        "total": n_samples,
        "all_votes": dict(answer_counts)
    }


# آزمایش Self-Consistency
sc_question = """
یک فروشگاه ۲۰٪ تخفیف روی کالاهایی می‌دهد که قیمت اصلی‌شان ۵۰۰۰۰ تومان است.
سپس ۱۵٪ مالیات به قیمت تخفیف‌خورده اضافه می‌شود.
قیمت نهایی چقدر است؟
"""

sc_result = self_consistency_cot(sc_question, n_samples=3)

print(f"\n{'='*60}")
print(f"📊 نتایج Self-Consistency:")
print(f"{'='*60}")
print(f"🗳️ همه رأی‌ها: {sc_result['all_votes']}")
print(f"✅ جواب اکثریت ({sc_result['vote_count']}/{sc_result['total']}):\n{sc_result['vote_result']}")


🔄 در حال اجرای 3 نمونه...
  نمونه 1/3...⏱️ زمان پاسخ: 7.7s | مدل: llama3.1:8b
 → 46000
  نمونه 2/3...⏱️ زمان پاسخ: 5.0s | مدل: llama3.1:8b
 → 138
  نمونه 3/3...⏱️ زمان پاسخ: 5.1s | مدل: llama3.1:8b
 → 46000

📊 نتایج Self-Consistency:
🗳️ همه رأی‌ها: {'46000': 2, '138': 1}
✅ جواب اکثریت (2/3):
46000


In [43]:

sc_result = self_consistency_cot(sc_question, n_samples=3,temperature=0.2)

print(f"\n{'='*60}")
print(f"📊 نتایج Self-Consistency:")
print(f"{'='*60}")
print(f"🗳️ همه رأی‌ها: {sc_result['all_votes']}")
print(f"✅ جواب اکثریت ({sc_result['vote_count']}/{sc_result['total']}):\n{sc_result['vote_result']}")


🔄 در حال اجرای 3 نمونه...
  نمونه 1/3...⏱️ زمان پاسخ: 7.6s | مدل: llama3.1:8b
 → 46000
  نمونه 2/3...⏱️ زمان پاسخ: 4.8s | مدل: llama3.1:8b
 → 46000
  نمونه 3/3...⏱️ زمان پاسخ: 4.8s | مدل: llama3.1:8b
 → 56000

📊 نتایج Self-Consistency:
🗳️ همه رأی‌ها: {'46000': 2, '56000': 1}
✅ جواب اکثریت (2/3):
46000
